<a href="https://colab.research.google.com/github/mohamedhassan1987/mohamed-hassan-portfolio/blob/main/Copy_of_1_1_%5BOGGs%5D_To_SRT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# UNIFIED GOOGLE COLAB PIPELINE (V10 - COMPLETE DUAL-ACCOUNT + FALLBACK SEARCH):
# Paste ONLY the Google Drive Folder URL below and Run!
# ==============================================================================
import os
import re
import glob
import time
import shutil
import ctypes
import zipfile
import threading
import subprocess
from datetime import datetime

# --- 1) PASTE GOOGLE DRIVE INPUT FOLDER LINK HERE ---
SOURCE_FOLDER_URL = "https://drive.google.com/drive/u/2/folders/1u5h9qZ1xynY-oB2uaUjHlSTip7rM4DCU"
# ----------------------------------------------------

# --- 2) ZIP SCHEDULE INTERVAL (IN HOURS) ---
ZIP_EVERY_HOURS = 1
# -------------------------------------------

MODEL_SIZE = "large-v3"
LANGUAGE = "zh"
CHINESE_PROMPT = "以下是普通话的句子，请使用简体中文并加上准确的标点符号。"
MEDIA_EXTS = (".ogg", ".opus", ".mp3", ".m4a", ".wav", ".flac", ".mp4", ".mkv", ".webm", ".aac")
AUTO_RUN_ON_CPU_IF_NO_GPU = True
ZIP_INTERVAL_SECONDS = max(1, int(float(ZIP_EVERY_HOURS) * 3600))


def ensure_drive_mounted(mount_point="/content/drive", retries=3):
    """Mounts Google Drive only if not already mounted, with safe retries."""
    mydrive = os.path.join(mount_point, "MyDrive")
    if os.path.exists(mydrive):
        try:
            if os.listdir(mydrive) is not None:
                return True
        except Exception:
            pass

    from google.colab import drive
    for attempt in range(1, retries + 1):
        try:
            print(f"🔗 ربط Google Drive (محاولة {attempt}/{retries})...")
            drive.mount(mount_point, force_remount=False)
            if os.path.exists(mydrive):
                print("✅ تم ربط Google Drive بنجاح.")
                return True
        except Exception as e:
            print(f"⚠️ محاولة الربط {attempt} فشلت: {e}")
            time.sleep(2)
    return False


def extract_folder_id_from_url(url_or_id: str) -> str:
    """Extracts Drive folder ID from standard, /u/1/, or query URLs."""
    text = (url_or_id or "").strip()
    match = re.search(r"/folders/([a-zA-Z0-9_-]+)", text)
    if match:
        return match.group(1)
    match_id = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", text)
    if match_id:
        return match_id.group(1)
    return text


def get_drive_service():
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    return build("drive", "v3")


def resolve_drive_folder_path_from_url(folder_url: str, mydrive_root="/content/drive/MyDrive") -> str:
    """
    Resolves a Google Drive folder URL across owned, shortcut, and shared folders:
    1. Direct path under MyDrive (following shortcuts if needed).
    2. Walk/search under MyDrive by folder name.
    3. Fallback Drive API search across shared/all drives by ID or name.
    """
    folder_id = extract_folder_id_from_url(folder_url)
    if not folder_id:
        raise ValueError("❌ لم يتم إدخال رابط أو معرف مجلد صالح في SOURCE_FOLDER_URL.")

    service = get_drive_service()

    meta = service.files().get(
        fileId=folder_id,
        fields="id, name, mimeType, parents, shortcutDetails, shared",
        supportsAllDrives=True
    ).execute()

    if meta.get("mimeType") == "application/vnd.google-apps.shortcut" and meta.get("shortcutDetails"):
        target_id = meta["shortcutDetails"].get("targetId")
        if target_id:
            folder_id = target_id
            meta = service.files().get(
                fileId=folder_id,
                fields="id, name, mimeType, parents, shared",
                supportsAllDrives=True
            ).execute()

    target_name = meta["name"]

    # 1) بناء المسار المباشر تحت MyDrive عبر سلسلة parents
    parts = []
    current_id = folder_id
    visited = set()
    while current_id and current_id not in visited:
        visited.add(current_id)
        info = service.files().get(
            fileId=current_id,
            fields="id, name, parents",
            supportsAllDrives=True
        ).execute()
        parents = info.get("parents", [])
        if not parents:
            break
        parts.append(info["name"])
        current_id = parents[0]

    parts.reverse()
    if parts:
        direct_path = os.path.join(mydrive_root, *parts)
        if os.path.exists(direct_path):
            return direct_path

    # 2) البحث داخل MyDrive (يشمل الاختصارات المضافة إلى MyDrive)
    if os.path.exists(mydrive_root):
        for root, dirs, _ in os.walk(mydrive_root):
            if target_name in dirs:
                return os.path.join(root, target_name)

    # 3) خطة احتياطية: البحث العام في Drive API (بما في ذلك المجلدات المشتركة)
    query = (
        f"name = '{target_name.replace(chr(39), chr(92)+chr(39))}' "
        f"and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    )
    results = service.files().list(
        q=query,
        fields="files(id, name, parents, shared)",
        includeItemsFromAllDrives=True,
        supportsAllDrives=True,
        pageSize=20
    ).execute()
    matched_folders = results.get("files", [])

    if matched_folders and os.path.exists(mydrive_root):
        for root, dirs, _ in os.walk(mydrive_root):
            if target_name in dirs:
                return os.path.join(root, target_name)

    raise FileNotFoundError(
        f"❌ تعذر ربط '{target_name}' كمسار محلي تحت {mydrive_root}. "
        f"تأكد من عمل Share (Editor) ثم 'Add shortcut to Drive' داخل My Drive."
    )


def derive_sibling_folders(input_dir: str):
    """Derives sibling -[Done] and -[ZIP] folders next to the input folder."""
    parent_dir = os.path.dirname(input_dir)
    input_name = os.path.basename(input_dir)

    m = re.match(r"^(#\d+-)1-(.+)$", input_name)
    if m:
        prefix_num, rest = m.group(1), m.group(2)
        done_name = f"{prefix_num}2-{rest}-[Done]"
        zip_name = f"{prefix_num}3-{rest}-[ZIP]"
    else:
        done_name = f"{input_name}-[Done]"
        zip_name = f"{input_name}-[ZIP]"

    done_dir = os.path.join(parent_dir, done_name)
    zip_dir = os.path.join(parent_dir, zip_name)
    os.makedirs(done_dir, exist_ok=True)
    os.makedirs(zip_dir, exist_ok=True)
    return parent_dir, done_dir, zip_dir


def get_package_lib_dir(pkg):
    if getattr(pkg, "__file__", None):
        base = os.path.dirname(pkg.__file__)
    else:
        base = list(pkg.__path__)[0]
    return os.path.join(base, "lib")


def format_timestamp(seconds: float) -> str:
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    if millis == 1000:
        secs += 1
        millis = 0
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def move_pair_to_done(media_path: str, srt_path: str, done_dir: str):
    os.makedirs(done_dir, exist_ok=True)
    dest_media = os.path.join(done_dir, os.path.basename(media_path))
    dest_srt = os.path.join(done_dir, os.path.basename(srt_path))

    if os.path.exists(dest_media) or os.path.exists(dest_srt):
        stem = os.path.splitext(os.path.basename(media_path))[0]
        ext = os.path.splitext(media_path)[1]
        ts = int(time.time())
        dest_media = os.path.join(done_dir, f"{stem}_{ts}{ext}")
        dest_srt = os.path.join(done_dir, f"{stem}_{ts}.srt")

    shutil.move(media_path, dest_media)
    shutil.move(srt_path, dest_srt)
    return dest_media, dest_srt


def scan_pipeline(input_dir: str, done_dir: str):
    remaining_media = []
    in_place_srt_media = []
    all_srt_candidates = []

    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            for fname in sorted(files):
                fpath = os.path.join(root, fname)
                if fname.lower().endswith(".srt") and os.path.getsize(fpath) > 0:
                    all_srt_candidates.append((fpath, False))
                elif fname.lower().endswith(MEDIA_EXTS):
                    srt_here = os.path.splitext(fpath)[0] + ".srt"
                    if os.path.exists(srt_here) and os.path.getsize(srt_here) > 0:
                        in_place_srt_media.append((fpath, srt_here))
                    else:
                        remaining_media.append(fpath)

    done_media = []
    if os.path.exists(done_dir):
        for root, _, files in os.walk(done_dir):
            for fname in sorted(files):
                fpath = os.path.join(root, fname)
                if fname.lower().endswith(".srt") and os.path.getsize(fpath) > 0:
                    all_srt_candidates.append((fpath, True))
                elif fname.lower().endswith(MEDIA_EXTS):
                    done_media.append(fpath)

    dedup_srt_map = {}
    for srt_path, is_in_done in all_srt_candidates:
        bname = os.path.basename(srt_path)
        if bname not in dedup_srt_map:
            dedup_srt_map[bname] = (srt_path, is_in_done)
        else:
            prev_path, prev_in_done = dedup_srt_map[bname]
            if (is_in_done and not prev_in_done) or (os.path.getmtime(srt_path) > os.path.getmtime(prev_path)):
                dedup_srt_map[bname] = (srt_path, is_in_done)

    unique_srt_files = [v[0] for v in dedup_srt_map.values()]
    return remaining_media, in_place_srt_media, done_media, unique_srt_files


def write_live_status_report(parent_dir: str, input_dir: str, done_dir: str, zip_dir: str):
    """Writes a live human-readable status summary file on Google Drive."""
    rem, _, done_m, uniq_srt = scan_pipeline(input_dir, done_dir)
    total = len(rem) + len(done_m)
    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    report_text = (
        f"LAST UPDATED : {now_str}\n"
        f"INPUT FOLDER : {os.path.basename(input_dir)}\n"
        f"DONE FOLDER  : {os.path.basename(done_dir)}\n"
        f"ZIP FOLDER   : {os.path.basename(zip_dir)}\n"
        f"----------------------------------------\n"
        f"TOTAL FILES  : {total}\n"
        f"COMPLETED    : {len(done_m)} media | {len(uniq_srt)} .srt\n"
        f"REMAINING    : {len(rem)}\n"
    )
    report_path = os.path.join(parent_dir, f"_STATUS_{os.path.basename(input_dir)}.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)


def create_hourly_srt_zip(input_dir: str, done_dir: str, zip_dir: str, zip_prefix="ogg-2-srt"):
    ensure_drive_mounted()
    _, _, _, unique_srt_files = scan_pipeline(input_dir, done_dir)
    if not unique_srt_files:
        return None

    os.makedirs(zip_dir, exist_ok=True)
    stamp = datetime.now().strftime("%d-%m_%H-%M")
    zip_filename = f"{zip_prefix}_{stamp}.zip"
    zip_drive_path = os.path.join(zip_dir, zip_filename)
    tmp_zip_path = zip_drive_path + ".tmp"

    added_names = set()
    with zipfile.ZipFile(tmp_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for srt_path in sorted(unique_srt_files):
            if not srt_path.lower().endswith(".srt"):
                continue
            arc_name = os.path.basename(srt_path)
            if arc_name in added_names:
                continue
            added_names.add(arc_name)
            zf.write(srt_path, arcname=arc_name)

    shutil.move(tmp_zip_path, zip_drive_path)
    print(f"\n📦 [ZIP] Saved {len(added_names)} .srt files -> {zip_drive_path}")
    return zip_drive_path


def start_hourly_zip_worker(input_dir: str, done_dir: str, zip_dir: str, stop_event, interval_sec: float):
    def _run():
        while not stop_event.wait(interval_sec):
            try:
                create_hourly_srt_zip(input_dir, done_dir, zip_dir)
            except Exception as e:
                print(f"\n⚠️ [ZIP] Error creating archive: {e}")

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    return t


if not ensure_drive_mounted():
    raise RuntimeError("❌ تعذر الوصول إلى Google Drive.")

INPUT_DIR = resolve_drive_folder_path_from_url(SOURCE_FOLDER_URL)
PARENT_DIR, DONE_DIR, ZIP_DIR = derive_sibling_folders(INPUT_DIR)

# ------------------------------------------------------------------------------
# STEP 1: INSTANT AUDIT & AUTO-MOVE ANY IN-PLACE COMPLETED PAIRS
# ------------------------------------------------------------------------------
remaining_media, in_place_srt_media, done_media, unique_srt_files = scan_pipeline(INPUT_DIR, DONE_DIR)

if in_place_srt_media:
    print(f"📦 Moving {len(in_place_srt_media)} already-paired files to '{os.path.basename(DONE_DIR)}'...")
    for m_path, s_path in in_place_srt_media:
        try:
            move_pair_to_done(m_path, s_path, DONE_DIR)
        except Exception as e:
            print(f"⚠️ Could not move {os.path.basename(m_path)}: {e}")
    remaining_media, in_place_srt_media, done_media, unique_srt_files = scan_pipeline(INPUT_DIR, DONE_DIR)

write_live_status_report(PARENT_DIR, INPUT_DIR, DONE_DIR, ZIP_DIR)

total_known_media = len(remaining_media) + len(done_media)
print("\n" + "=" * 72)
print("📊 QUICK WORKSPACE AUDIT (V10 - URL AUTO-DISCOVERY)")
print("=" * 72)
print(f"📂 Input Folder               : {INPUT_DIR}")
print(f"📁 Done Folder                : {DONE_DIR}")
print(f"🗜️ ZIP Folder                 : {ZIP_DIR}")
print(f"⏱️ ZIP Interval (Hours)       : {ZIP_EVERY_HOURS}")
print(f"🎵 Total Media Files          : {total_known_media}")
print(f"✅ Done & Moved (Media/SRT)   : {len(done_media)} / {len(unique_srt_files)}")
print(f"⏳ Remaining To Transcribe    : {len(remaining_media)}")
print("=" * 72 + "\n")

if unique_srt_files:
    create_hourly_srt_zip(INPUT_DIR, DONE_DIR, ZIP_DIR)

stop_zip_event = threading.Event()
start_hourly_zip_worker(INPUT_DIR, DONE_DIR, ZIP_DIR, stop_zip_event, ZIP_INTERVAL_SECONDS)

# ------------------------------------------------------------------------------
# STEP 2: AUTO-SELECT HARDWARE (GPU float16 FIRST -> CPU int8 FALLBACK)
# ------------------------------------------------------------------------------
import torch

try:
    if torch.cuda.is_available():
        device = "cuda"
        compute_type = "float16"
        print("⚡ GPU متاح! سيتم التشغيل على CUDA (float16).")
    elif AUTO_RUN_ON_CPU_IF_NO_GPU:
        device = "cpu"
        compute_type = "int8"
        print("⚠️ GPU غير متاح حالياً — سيتم الاستمرار تلقائياً على CPU (int8) بنفس نموذج large-v3.")
    else:
        device = None
        compute_type = None
        print("ℹ️ GPU غير متاح وتم إيقاف التشغيل التلقائي.")

    if device is not None and len(remaining_media) == 0:
        print("🎉 جميع الملفات تم تفريغها ونقلها إلى مجلد المنجز بالفعل!")
        device = None

    if device is not None:
        print("⏳ Installing faster-whisper, audio libraries & runtime dependencies...")
        subprocess.run(
            ["pip", "install", "-q", "faster-whisper", "soundfile", "librosa", "nvidia-cublas-cu12", "nvidia-cudnn-cu12"],
            check=True
        )

        if device == "cuda":
            import nvidia.cublas
            import nvidia.cudnn

            cuda_lib_dirs = [
                d for d in (get_package_lib_dir(nvidia.cublas), get_package_lib_dir(nvidia.cudnn))
                if os.path.isdir(d)
            ]

            if cuda_lib_dirs:
                with open("/etc/ld.so.conf.d/nvidia_cu12.conf", "w") as f:
                    f.write("\n".join(cuda_lib_dirs) + "\n")
                subprocess.run(["ldconfig"], check=False)
                os.environ["LD_LIBRARY_PATH"] = ":".join(cuda_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
                for lib_dir in cuda_lib_dirs:
                    for so_file in sorted(glob.glob(os.path.join(lib_dir, "*.so*"))):
                        try:
                            ctypes.CDLL(so_file, mode=ctypes.RTLD_GLOBAL)
                        except OSError:
                            pass

        import librosa
        from faster_whisper import WhisperModel

        print(f"🚀 Loading Whisper '{MODEL_SIZE}' [Language: {LANGUAGE.upper()}] on {device.upper()} ({compute_type})...")
        model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)

        total_rem = len(remaining_media)
        processed_now = 0
        errors_now = 0
        start_all = time.time()

        try:
            for idx, media_path in enumerate(remaining_media, start=1):
                ensure_drive_mounted()
                base_path_no_ext = os.path.splitext(media_path)[0]
                base_name = os.path.basename(base_path_no_ext)
                srt_temp_next_to_media = base_path_no_ext + ".srt"
                expected_srt_in_done = os.path.join(DONE_DIR, base_name + ".srt")

                if os.path.exists(expected_srt_in_done) and os.path.getsize(expected_srt_in_done) > 0:
                    continue

                t0 = time.time()
                ext = os.path.splitext(media_path)[1]
                local_tmp_media = f"/content/tmp_in_{idx}{ext}"
                local_tmp_srt = f"/content/tmp_out_{idx}.srt"

                try:
                    shutil.copy2(media_path, local_tmp_media)
                    audio_array, _ = librosa.load(local_tmp_media, sr=16000, mono=True)

                    segments, info = model.transcribe(
                        audio_array,
                        language=LANGUAGE,
                        initial_prompt=CHINESE_PROMPT,
                        beam_size=5,
                        condition_on_previous_text=False,
                        vad_filter=True,
                        vad_parameters=dict(min_silence_duration_ms=500)
                    )

                    srt_lines = []
                    for seg_i, seg in enumerate(segments, start=1):
                        text = seg.text.strip()
                        if text:
                            srt_lines.append(
                                f"{seg_i}\n{format_timestamp(seg.start)} --> {format_timestamp(seg.end)}\n{text}\n"
                            )

                    with open(local_tmp_srt, "w", encoding="utf-8") as f:
                        f.write("\n".join(srt_lines))

                    if os.path.exists(local_tmp_srt) and os.path.getsize(local_tmp_srt) > 0:
                        shutil.copy2(local_tmp_srt, srt_temp_next_to_media)
                        move_pair_to_done(media_path, srt_temp_next_to_media, DONE_DIR)
                        processed_now += 1
                        elapsed = time.time() - t0
                        print(f"✅ [{idx}/{total_rem}] ({device.upper()}) {base_name} -> Moved to '{os.path.basename(DONE_DIR)}' | {elapsed:.1f}s")
                        if processed_now % 5 == 0:
                            write_live_status_report(PARENT_DIR, INPUT_DIR, DONE_DIR, ZIP_DIR)
                    else:
                        errors_now += 1
                        print(f"⚠️ [{idx}/{total_rem}] Empty SRT generated for '{base_name}', kept in input folder.")

                except Exception as e:
                    errors_now += 1
                    print(f"❌ [{idx}/{total_rem}] Error in '{base_name}': {e}")

                finally:
                    if os.path.exists(local_tmp_media):
                        os.remove(local_tmp_media)
                    if os.path.exists(local_tmp_srt):
                        os.remove(local_tmp_srt)

        except KeyboardInterrupt:
            print("\n⏹️ تم إيقاف التشغيل يدوياً بواسطة المستخدم بأمان.")

        create_hourly_srt_zip(INPUT_DIR, DONE_DIR, ZIP_DIR)
        write_live_status_report(PARENT_DIR, INPUT_DIR, DONE_DIR, ZIP_DIR)

        _, _, final_done_media, final_done_srt = scan_pipeline(INPUT_DIR, DONE_DIR)
        elapsed_min = (time.time() - start_all) / 60.0
        print("\n" + "=" * 72)
        print("🎉 SESSION SUMMARY")
        print(f"   Device Used           : {device.upper()} ({compute_type})")
        print(f"   Processed & Moved Now : {processed_now}")
        print(f"   Total Done Media/SRT  : {len(final_done_media)} / {len(final_done_srt)}")
        print(f"   Errors                : {errors_now}")
        print(f"   Elapsed Time          : {elapsed_min:.2f} minutes")
        print("=" * 72)

finally:
    stop_zip_event.set()


📊 QUICK WORKSPACE AUDIT (V10 - URL AUTO-DISCOVERY)
📂 Input Folder               : /content/drive/MyDrive/#1-1-[OGGs]-To-SRT
📁 Done Folder                : /content/drive/MyDrive/#1-2-[OGGs]-To-SRT-[Done]
🗜️ ZIP Folder                 : /content/drive/MyDrive/#1-3-[OGGs]-To-SRT-[ZIP]
⏱️ ZIP Interval (Hours)       : 1
🎵 Total Media Files          : 161
✅ Done & Moved (Media/SRT)   : 0 / 0
⏳ Remaining To Transcribe    : 161

⚡ GPU متاح! سيتم التشغيل على CUDA (float16).
⏳ Installing faster-whisper, audio libraries & runtime dependencies...
🚀 Loading Whisper 'large-v3' [Language: ZH] on CUDA (float16)...


✅ [1/161] (CUDA) 48-Chinesewithmaya -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 76.1s
✅ [2/161] (CUDA) 48-Chloechinesecoach -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 7.2s
✅ [3/161] (CUDA) 48-Everyday Chinese-(12) -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 2.1s
✅ [4/161] (CUDA) 48-GraceMandarinChinese -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 39.1s
✅ [5/161] (CUDA) 48-ShuoshuoChinese -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 51.3s
✅ [6/161] (CUDA) 49-Chinese with Ben-(5) -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 1.3s
✅ [7/161] (CUDA) 49-Chinesewithmaya -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 36.9s
✅ [8/161] (CUDA) 49-Everyday Chinese-(12) -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 1.0s
✅ [9/161] (CUDA) 49-GraceMandarinChinese -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 46.2s
✅ [10/161] (CUDA) 49-ShuoshuoChinese -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 23.8s
✅ [11/161] (CUDA) 50-Chinese with Ben-(5) -> Moved to '#1-2-[OGGs]-To-SRT-[Done]' | 1.9s
✅ [12/161] (CUDA) 50-ChineseWithJingD -> Move